# 🌊 Ripple Engine — 100 Effects

Run **Cell 1 → Cell 2 → Cell 3** in order (Shift+Enter).
The interactive control panel appears below Cell 3.

**Bugs fixed in this version:**
- Cell 3 now correctly uses effects loaded by Cell 2 (was referencing undefined `EFFECTS_SOURCE`)
- Effect #100 (Glitch Ripple) ID now parsed correctly (was returning ID 10 instead of 100)
- Explicit `import math` added to Cell 3 for safety

### Cell 1 · Install dependencies

In [ ]:
!pip install opencv-python-headless tqdm numpy ipywidgets -q

### Cell 2 · Load the 100-effect engine
This defines all 100 effect functions and the `EFFECTS` dispatch table globally.

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — 100 Ripple Effects Engine
# Run this cell ONCE. It defines all 100 effect functions globally.
# ════════════════════════════════════════════════════════════════
import numpy as np, cv2, math, random

EFFECTS_SRC = r'''


import numpy as np
import cv2
import math
import random

# ─────────────────────────────────────────────────────────────────────────────
# LOW-LEVEL HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _roll_rows(frame, shifts):
    out = np.empty_like(frame)
    for y, s in enumerate(shifts):
        out[y] = np.roll(frame[y], int(s), axis=0)
    return out

def _roll_cols(frame, shifts):
    out = np.empty_like(frame)
    for x, s in enumerate(shifts):
        out[:, x] = np.roll(frame[:, x], int(s), axis=0)
    return out

def _remap(frame, map_x, map_y):
    return cv2.remap(
        frame, map_x.astype(np.float32), map_y.astype(np.float32),
        cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT
    )

def _meshgrid(h, w):
    Y, X = np.mgrid[0:h, 0:w].astype(np.float32)
    return X, Y

def _norm(frame):
    return np.clip(frame, 0, 255)

def _radial(h, w, cx=None, cy=None):
    """Return distance and angle maps from centre."""
    cx = cx if cx is not None else w / 2
    cy = cy if cy is not None else h / 2
    X, Y = _meshgrid(h, w)
    D = np.sqrt((X - cx)**2 + (Y - cy)**2)
    A = np.arctan2(Y - cy, X - cx)
    return D, A

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 1 — BASIC RIPPLE (1-10) ════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def ripple(frame, t, p):
    """1. Ripple — standard sine wave shift per row."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    return _roll_rows(frame, s)

def gentle_ripple(frame, t, p):
    """2. Gentle Ripple — low-amplitude smooth wave."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s = (p["strength"]*0.4) * np.sin(2*np.pi*(rows/(p["wave_len"]*1.5) + t*(p["speed"]*0.6)))
    return _roll_rows(frame, s)

def strong_ripple(frame, t, p):
    """3. Strong Ripple — high-amplitude, fast."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s = (p["strength"]*2.5) * np.sin(2*np.pi*(rows/(p["wave_len"]*0.6) + t*(p["speed"]*1.8)))
    s += (p["strength"]*0.8) * np.sin(4*np.pi*(rows/(p["wave_len"]*0.6) + t*p["speed"]))
    return _roll_rows(frame, s)

def soft_ripple(frame, t, p):
    """4. Soft Ripple — Gaussian-smoothed displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    raw = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    raw = cv2.GaussianBlur(raw.reshape(-1,1), (1,15), 5).flatten()
    return _roll_rows(frame, raw)

def hard_ripple(frame, t, p):
    """5. Hard Ripple — square-wave clipped displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    sine = np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    s = np.sign(sine) * p["strength"]
    return _roll_rows(frame, s)

def smooth_ripple(frame, t, p):
    """6. Smooth Ripple — smoothstep eased sine."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    raw  = (np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"])) + 1) / 2  # 0→1
    eased = raw*raw*(3 - 2*raw)  # smoothstep
    s = (eased*2 - 1) * p["strength"]
    return _roll_rows(frame, s)

def sharp_ripple(frame, t, p):
    """7. Sharp Ripple — sawtooth wave displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    phase = (rows/p["wave_len"] + t*p["speed"]) % 1.0
    s = (phase*2 - 1) * p["strength"]
    return _roll_rows(frame, s)

def random_ripple(frame, t, p):
    """8. Random Ripple — per-row random shift with temporal seed."""
    h, w = frame.shape[:2]
    rng = np.random.default_rng(int(t * 30) % 10000)
    s = rng.uniform(-p["strength"], p["strength"], h)
    s = cv2.GaussianBlur(s.reshape(-1,1).astype(np.float32), (1,9), 3).flatten()
    return _roll_rows(frame, s)

def multi_ripple(frame, t, p):
    """9. Multi Ripple — 3 sine waves summed."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    wl = p["wave_len"]
    sp = p["speed"]
    st = p["strength"]
    s  = (st*0.6)*np.sin(2*np.pi*(rows/wl       + t*sp)) \
       + (st*0.3)*np.sin(2*np.pi*(rows/(wl*0.5)  + t*sp*1.5)) \
       + (st*0.1)*np.sin(2*np.pi*(rows/(wl*0.25) + t*sp*3))
    return _roll_rows(frame, s)

def continuous_ripple(frame, t, p):
    """10. Continuous Ripple — seamless loop via both axes."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    cols = np.arange(w, dtype=np.float32)
    sr = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    sc = p["strength"]*0.5 * np.sin(2*np.pi*(cols/p["wave_len"] + t*p["speed"]*0.7))
    out = _roll_rows(frame, sr)
    return _roll_cols(out, sc)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 2 — WATER RIPPLE (11-20) ═══════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def water_ripple(frame, t, p):
    """11. Water Ripple — radial expanding ring from centre."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, _ = _radial(h, w)
    disp  = p["strength"] * np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"]))
    decay = np.exp(-D / (max(h,w)*0.5))
    disp *= decay
    ang   = np.arctan2(Y - h/2, X - w/2)
    mx = X + disp * np.cos(ang)
    my = Y + disp * np.sin(ang)
    return _remap(frame, mx, my)

def pond_ripple(frame, t, p):
    """12. Pond Ripple — multiple concentric rings, slow decay."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, _ = _radial(h, w)
    ang   = np.arctan2(Y - h/2, X - w/2)
    disp  = p["strength"] * np.sin(2*np.pi*(D/(p["wave_len"]*1.2) - t*(p["speed"]*0.6)))
    disp *= np.exp(-D / (max(h,w)*0.8))
    mx = X + disp * np.cos(ang)
    my = Y + disp * np.sin(ang)
    return _remap(frame, mx, my)

def ocean_ripple(frame, t, p):
    """13. Ocean Ripple — large rolling wave + secondary chop."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s  = (p["strength"]*2)   * np.sin(2*np.pi*(rows/(p["wave_len"]*2) + t*(p["speed"]*0.5)))
    s += (p["strength"]*0.5) * np.sin(2*np.pi*(rows/(p["wave_len"]*0.4)+ t*(p["speed"]*1.2)))
    s += (p["strength"]*0.2) * np.sin(2*np.pi*(rows/(p["wave_len"]*0.15)+ t*(p["speed"]*2.5)))
    return _roll_rows(frame, s)

def lake_ripple(frame, t, p):
    """14. Lake Ripple — radial + gentle horizontal sway."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    disp  = p["strength"]*0.8 * np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"]*0.4))
    disp *= np.exp(-D / (max(h,w)*0.7))
    mx = X + disp * np.cos(ang)
    my = Y + disp * np.sin(ang)
    out = _remap(frame, mx, my)
    rows = np.arange(h, dtype=np.float32)
    sc = p["strength"]*0.3 * np.sin(2*np.pi*(rows/(p["wave_len"]*3) + t*p["speed"]*0.3))
    return _roll_rows(out, sc)

def river_ripple(frame, t, p):
    """15. River Ripple — horizontal flow + vertical undulation."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    cols = np.arange(w, dtype=np.float32)
    sr = p["strength"]*0.4 * np.sin(2*np.pi*(rows/(p["wave_len"]*2) + t*p["speed"]*0.3))
    sc = p["strength"]     * np.sin(2*np.pi*(cols/p["wave_len"]      + t*p["speed"]))
    out = _roll_rows(frame, sr)
    return _roll_cols(out, sc)

def rain_ripple(frame, t, p):
    """16. Rain Ripple — many small radial rings at random positions."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    disp = np.zeros((h, w), dtype=np.float32)
    rng  = np.random.default_rng(42)
    n_drops = 8
    cx_arr = rng.uniform(0.1, 0.9, n_drops) * w
    cy_arr = rng.uniform(0.1, 0.9, n_drops) * h
    for i, (cx, cy) in enumerate(zip(cx_arr, cy_arr)):
        phase  = (t*p["speed"] + i*0.37) % 1.0
        D      = np.sqrt((X-cx)**2 + (Y-cy)**2)
        ring_r = phase * max(h,w) * 0.4
        mask   = np.exp(-((D - ring_r)**2) / (p["wave_len"]**2))
        disp  += p["strength"] * mask * np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"]))
    ang = np.arctan2(Y - h/2, X - w/2)
    mx  = X + disp * np.cos(np.arctan2(Y - cy_arr[0], X - cx_arr[0]))
    my  = Y + disp * np.sin(np.arctan2(Y - cy_arr[0], X - cx_arr[0]))
    return _remap(frame, mx, my)

def droplet_ripple(frame, t, p):
    """17. Droplet Ripple — single-point expanding circular ripple."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    cx, cy = w*0.5, h*0.5
    D   = np.sqrt((X-cx)**2 + (Y-cy)**2) + 1e-6
    ang = np.arctan2(Y-cy, X-cx)
    ring_r = (t*p["speed"]*max(h,w)*0.3) % (max(h,w)*0.5)
    mask   = np.exp(-((D - ring_r)**2) / (p["wave_len"]**2))
    disp   = p["strength"] * mask
    mx = X + disp*np.cos(ang)
    my = Y + disp*np.sin(ang)
    return _remap(frame, mx, my)

def splash_ripple(frame, t, p):
    """18. Splash Ripple — burst ripple with fast initial expansion."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    cx, cy = w*0.5, h*0.5
    D   = np.sqrt((X-cx)**2 + (Y-cy)**2) + 1e-6
    ang = np.arctan2(Y-cy, X-cx)
    r   = (t*p["speed"]*max(h,w)*0.5) % (max(h,w)*0.6)
    burst = np.exp(-((D-r)**2)/(p["wave_len"]**2)) * np.exp(-t*0.5)
    disp  = p["strength"]*1.5 * burst
    mx = X + disp*np.cos(ang)
    my = Y + disp*np.sin(ang)
    return _remap(frame, mx, my)

def bubble_ripple(frame, t, p):
    """19. Bubble Ripple — multiple small circular bubbles rising."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    disp_x = np.zeros((h,w), dtype=np.float32)
    disp_y = np.zeros((h,w), dtype=np.float32)
    rng = np.random.default_rng(7)
    for i in range(6):
        cx = rng.uniform(0.1,0.9)*w
        cy = ((h - (t*p["speed"]*h*0.3 + i*(h/6))) % h)
        r  = rng.uniform(20,60)
        D  = np.sqrt((X-cx)**2 + (Y-cy)**2) + 1e-6
        mask = np.exp(-(D/r)**2)
        ang  = np.arctan2(Y-cy, X-cx)
        disp_x += p["strength"]*0.5 * mask * np.cos(ang) * np.sin(D/p["wave_len"]*4)
        disp_y += p["strength"]*0.5 * mask * np.sin(ang) * np.sin(D/p["wave_len"]*4)
    return _remap(frame, X+disp_x, Y+disp_y)

def wet_ripple(frame, t, p):
    """20. Wet Ripple — blurred + displaced, like reflection in shallow water."""
    h, w = frame.shape[:2]
    blurred = cv2.GaussianBlur(frame, (7,7), 2)
    rows    = np.arange(h, dtype=np.float32)
    s = p["strength"]*0.6 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    moved  = _roll_rows(blurred, s)
    alpha  = 0.6 + 0.3*np.sin(t*p["speed"])
    return _norm(frame * (1-alpha) + moved * alpha)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 3 — DISTORTION RIPPLE (21-30) ══════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def ripple_warp(frame, t, p):
    """21. Ripple Warp — 2D sine warp in both X and Y."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = p["strength"] * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]))
    dy = p["strength"]*0.7 * np.sin(2*np.pi*(X/p["wave_len"] + t*p["speed"]*0.8))
    return _remap(frame, X+dx, Y+dy)

def ripple_distortion(frame, t, p):
    """22. Ripple Distortion — phase-shifted XY warp."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = p["strength"] * np.sin(2*np.pi*Y/p["wave_len"] + t*p["speed"]*2)
    dy = p["strength"] * np.cos(2*np.pi*X/p["wave_len"] + t*p["speed"]*2)
    return _remap(frame, X+dx, Y+dy)

def wave_distortion(frame, t, p):
    """23. Wave Distortion — diagonal travelling wave."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    diag = (X/w + Y/h)
    dx = p["strength"]*1.2 * np.sin(2*np.pi*(diag*3 + t*p["speed"]))
    dy = p["strength"]*0.8 * np.cos(2*np.pi*(diag*3 + t*p["speed"]))
    return _remap(frame, X+dx, Y+dy)

def turbulent_ripple(frame, t, p):
    """24. Turbulent Ripple — layered noise-like multi-freq warp."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = (p["strength"]   * np.sin(2*np.pi*(Y/p["wave_len"]       + t*p["speed"]))
        + p["strength"]*0.5 * np.sin(2*np.pi*(Y/(p["wave_len"]*0.5) + X/(p["wave_len"]*2) + t*p["speed"]*1.7))
        + p["strength"]*0.25* np.sin(2*np.pi*(Y/(p["wave_len"]*0.25)+ t*p["speed"]*3.1)))
    dy = (p["strength"]   * np.cos(2*np.pi*(X/p["wave_len"]       + t*p["speed"]*0.9))
        + p["strength"]*0.5 * np.cos(2*np.pi*(X/(p["wave_len"]*0.5)+ Y/(p["wave_len"]*3) + t*p["speed"]*1.3)))
    return _remap(frame, X+dx, Y+dy)

def twisted_ripple(frame, t, p):
    """25. Twisted Ripple — rotation angle modulated by radius."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    twist   = p["strength"] * 0.002 * D * np.sin(t*p["speed"])
    new_ang = ang + twist
    mx = w/2 + D*np.cos(new_ang)
    my = h/2 + D*np.sin(new_ang)
    return _remap(frame, mx, my)

def curved_ripple(frame, t, p):
    """26. Curved Ripple — cubic Bezier-inspired row offsets."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32) / h  # 0→1
    s = p["strength"] * (rows**3 - rows) * np.sin(t*p["speed"]*2*np.pi)
    return _roll_rows(frame, s * h * 0.1)

def swirl_ripple(frame, t, p):
    """27. Swirl Ripple — polar-coordinate swirl + radial sine."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    swirl  = p["strength"] * 0.003 * np.sin(D/p["wave_len"]*2 - t*p["speed"]*2)
    new_ang = ang + swirl
    mx = w/2 + D*np.cos(new_ang)
    my = h/2 + D*np.sin(new_ang)
    return _remap(frame, mx, my)

def liquid_ripple(frame, t, p):
    """28. Liquid Ripple — slow viscous flow simulation."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = p["strength"] * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]*0.4)) \
       * np.cos(2*np.pi*(X/(p["wave_len"]*1.5) + t*p["speed"]*0.3))
    dy = p["strength"]*0.6 * np.cos(2*np.pi*(X/p["wave_len"] + t*p["speed"]*0.4)) \
       * np.sin(2*np.pi*(Y/(p["wave_len"]*1.5) + t*p["speed"]*0.3))
    return _remap(frame, X+dx, Y+dy)

def jelly_ripple(frame, t, p):
    """29. Jelly Ripple — bouncy elastic overshoot motion."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    decay  = np.exp(-rows/h*2)
    bounce = np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"])) * decay
    spring = np.sin(2*np.pi*(rows/(p["wave_len"]*0.3) + t*p["speed"]*3)) * (1-decay) * 0.3
    s = p["strength"] * (bounce + spring)
    return _roll_rows(frame, s)

def elastic_ripple(frame, t, p):
    """30. Elastic Ripple — elastic-eased ping-pong displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    raw  = np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    # elastic ease
    k    = 2*np.pi/3
    sign = np.sign(raw)
    mag  = np.abs(raw)
    elas = sign * (-(2**(10*(mag-1))) * np.sin((mag*10 - 10.75)*k))
    elas = np.nan_to_num(elas, 0)
    return _roll_rows(frame, elas * p["strength"])

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 4 — ENERGY RIPPLE (31-40) ══════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def energy_ripple(frame, t, p):
    """31. Energy Ripple — glowing radial pulse with brightness boost."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    disp   = p["strength"] * np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"]))
    mx = X + disp*np.cos(ang)
    my = Y + disp*np.sin(ang)
    out = _remap(frame, mx, my)
    glow = 1 + 0.15*np.sin(t*p["speed"]*2*np.pi)
    return _norm(out * glow)

def plasma_ripple(frame, t, p):
    """32. Plasma Ripple — colour-rotated multi-sine interference."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    v  = (np.sin(X/p["wave_len"] + t*p["speed"])
        + np.sin(Y/p["wave_len"] + t*p["speed"])
        + np.sin((X+Y)/(p["wave_len"]*1.4) + t*p["speed"])) / 3.0  # -1→1
    dx = p["strength"] * v
    dy = p["strength"] * np.roll(v, p["wave_len"]//2)
    out = _remap(frame, X+dx, Y+dy)
    tint = np.array([0.15,0,0.25], dtype=np.float32)
    return _norm(out + v[...,None]*p["strength"]*tint*255)

def electric_ripple(frame, t, p):
    """33. Electric Ripple — jagged high-freq horizontal jitter."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s  = p["strength"]*1.5 * np.sin(2*np.pi*(rows/(p["wave_len"]*0.2) + t*p["speed"]*3))
    s += p["strength"]*0.5 * (np.random.rand(h)*2-1)  # static noise
    return _roll_rows(frame, s)

def lightning_ripple(frame, t, p):
    """34. Lightning Ripple — single zigzag bolt displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    bolt_phase = (t * p["speed"] * 5) % 1.0
    bolt_y     = bolt_phase * h
    dist       = np.abs(rows - bolt_y)
    mask       = np.exp(-(dist**2) / (p["wave_len"]**2))
    zigzag     = np.sign(np.sin(rows / (p["wave_len"]*0.1)))
    s = p["strength"]*2 * mask * zigzag
    return _roll_rows(frame, s)

def force_ripple(frame, t, p):
    """35. Force Ripple — centrifugal outward push from centre."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    force  = p["strength"] * np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"]))
    push   = force * (1 - D/max(D.max(),1))
    mx = X + push*np.cos(ang)
    my = Y + push*np.sin(ang)
    return _remap(frame, mx, my)

def power_ripple(frame, t, p):
    """36. Power Ripple — exponential amplitude ramp from top."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    ramp = np.exp(rows/h * 2) / np.exp(2)
    s    = p["strength"]*1.5 * ramp * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    return _roll_rows(frame, s)

def shockwave_ripple(frame, t, p):
    """37. Shockwave Ripple — single expanding ring shockfront."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    ring_r = (t * p["speed"] * max(h,w) * 0.4) % (max(h,w)*0.5)
    mask   = np.exp(-((D - ring_r)**2) / (p["wave_len"]**2 * 0.5))
    disp   = p["strength"]*2 * mask
    mx = X + disp*np.cos(ang)
    my = Y + disp*np.sin(ang)
    return _remap(frame, mx, my)

def pulse_ripple(frame, t, p):
    """38. Pulse Ripple — heartbeat-like radial throb."""
    h, w = frame.shape[:2]
    beat  = abs(np.sin(t * p["speed"] * np.pi))
    X, Y  = _meshgrid(h, w)
    D, ang = _radial(h, w)
    disp  = p["strength"] * beat * np.exp(-D / (max(h,w)*0.3))
    mx = X + disp*np.cos(ang)
    my = Y + disp*np.sin(ang)
    out = _remap(frame, mx, my)
    return _norm(out * (1 + 0.1*beat))

def quantum_ripple(frame, t, p):
    """39. Quantum Ripple — probability-wave interference pattern."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    psi1 = np.sin(2*np.pi*(X/p["wave_len"] + t*p["speed"]))
    psi2 = np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]*0.7 + np.pi/3))
    psi3 = np.sin(2*np.pi*((X+Y)/(p["wave_len"]*1.4) + t*p["speed"]*1.3))
    prob = (psi1*psi2*psi3)
    dx = p["strength"] * prob
    dy = p["strength"] * np.roll(prob, p["wave_len"]//3, axis=0)
    return _remap(frame, X+dx, Y+dy)

def aura_ripple(frame, t, p):
    """40. Aura Ripple — soft glowing radial halo pulse."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    r_norm = D / (max(h,w)*0.5)
    halo   = np.exp(-((r_norm - 0.5 - 0.1*np.sin(t*p["speed"]*2))**2) / 0.05)
    disp   = p["strength"] * halo
    mx = X + disp*np.cos(ang)
    my = Y + disp*np.sin(ang)
    out = _remap(frame, mx, my)
    return _norm(out + halo[...,None]*20)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 5 — SONIC RIPPLE (41-50) ═══════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def sonic_ripple(frame, t, p):
    """41. Sonic Ripple — speaker-cone radiating pressure waves."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    wave  = np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"])) / (D/50 + 1)
    mx = X + p["strength"]*wave*np.cos(ang)
    my = Y + p["strength"]*wave*np.sin(ang)
    return _remap(frame, mx, my)

def audio_ripple(frame, t, p):
    """42. Audio Ripple — stereo left/right channel simulation."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    cols = np.arange(w, dtype=np.float32)
    left  = p["strength"]*0.8 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    right = p["strength"]*0.8 * np.sin(2*np.pi*(rows/p["wave_len"] - t*p["speed"] + np.pi))
    s     = np.where(cols[None,:] < w//2, left[:,None], right[:,None]).mean(axis=1)
    return _roll_rows(frame, s)

def beat_ripple(frame, t, p):
    """43. Beat Ripple — sharp kick-drum thump on beat."""
    h, w  = frame.shape[:2]
    beat_t = (t * p["speed"] * 2) % 1.0
    impulse = np.exp(-beat_t * 8)
    X, Y   = _meshgrid(h, w)
    D, ang = _radial(h, w)
    disp   = p["strength"]*2 * impulse * np.exp(-D / (max(h,w)*0.4))
    mx = X + disp*np.cos(ang)
    my = Y + disp*np.sin(ang)
    return _remap(frame, mx, my)

def music_ripple(frame, t, p):
    """44. Music Ripple — multi-harmonic waveform (5 harmonics)."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    harmonics = [1, 2, 3, 5, 8]
    weights   = [0.5, 0.25, 0.125, 0.075, 0.05]
    s = sum(w_*np.sin(2*np.pi*(rows/(p["wave_len"]/n) + t*p["speed"]*n))
            for n, w_ in zip(harmonics, weights))
    return _roll_rows(frame, s * p["strength"])

def frequency_ripple(frame, t, p):
    """45. Frequency Ripple — sweeping frequency modulation."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    freq = 1/p["wave_len"] * (1 + 0.5*np.sin(t*p["speed"]))
    s    = p["strength"] * np.sin(2*np.pi*(rows*freq + t*p["speed"]))
    return _roll_rows(frame, s)

def bass_ripple(frame, t, p):
    """46. Bass Ripple — low-freq slow deep displacement."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = p["strength"]*1.5 * np.sin(2*np.pi*(Y/(p["wave_len"]*3) + t*p["speed"]*0.3))
    dy = p["strength"]     * np.sin(2*np.pi*(X/(p["wave_len"]*3) + t*p["speed"]*0.2))
    return _remap(frame, X+dx, Y+dy)

def echo_ripple(frame, t, p):
    """47. Echo Ripple — decaying repeated copies of the frame."""
    h, w = frame.shape[:2]
    rows  = np.arange(h, dtype=np.float32)
    out   = frame.copy()
    for i in range(1, 4):
        delay = i * 0.18
        s     = p["strength"]*(0.5**i) * np.sin(2*np.pi*(rows/p["wave_len"] + (t-delay)*p["speed"]))
        echo  = _roll_rows(frame, s)
        out   = _norm(out * 0.7 + echo * 0.3)
    return out

def vibration_ripple(frame, t, p):
    """48. Vibration Ripple — high-freq small-amplitude jitter."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"]*0.3 * np.sin(2*np.pi*(rows/(p["wave_len"]*0.1) + t*p["speed"]*8))
    s   += p["strength"]*0.3 * np.sin(2*np.pi*(rows/(p["wave_len"]*0.07)+ t*p["speed"]*11))
    return _roll_rows(frame, s)

def resonance_ripple(frame, t, p):
    """49. Resonance Ripple — standing wave (left+right interference)."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    forward  = np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    backward = np.sin(2*np.pi*(rows/p["wave_len"] - t*p["speed"]))
    s        = p["strength"] * (forward + backward) * 0.5
    return _roll_rows(frame, s)

def soundwave_ripple(frame, t, p):
    """50. Soundwave Ripple — longitudinal compression bands."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    compress = 1 + (p["strength"]/w) * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    X, Y = _meshgrid(h, w)
    mx   = X * compress[:, None]
    return _remap(frame, np.clip(mx, 0, w-1), Y)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 6 — MOTION RIPPLE (51-60) ══════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def shaky_ripple(frame, t, p):
    """51. Shaky Ripple — random frame shake + ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out = _roll_rows(frame, s)
    dx  = int(np.random.uniform(-p["strength"]*0.3, p["strength"]*0.3))
    dy  = int(np.random.uniform(-p["strength"]*0.3, p["strength"]*0.3))
    return np.roll(np.roll(out, dx, axis=1), dy, axis=0)

def wiggle_ripple(frame, t, p):
    """52. Wiggle Ripple — sinusoidal X+Y wiggle simultaneously."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = p["strength"] * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"])) * np.cos(t*2)
    dy = p["strength"] * np.cos(2*np.pi*(X/p["wave_len"] + t*p["speed"])) * np.sin(t*2)
    return _remap(frame, X+dx, Y+dy)

def jitter_ripple(frame, t, p):
    """53. Jitter Ripple — block-level random displacement."""
    h, w  = frame.shape[:2]
    out   = frame.copy()
    blk   = max(4, p["wave_len"]//4)
    rng   = np.random.default_rng(int(t*24))
    for y in range(0, h, blk):
        dx = int(rng.uniform(-p["strength"], p["strength"]))
        out[y:y+blk] = np.roll(out[y:y+blk], dx, axis=1)
    return out

def camera_ripple(frame, t, p):
    """54. Camera Ripple — handheld camera sway + ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    sway_x = int(p["strength"]*0.5*np.sin(t*p["speed"]*0.7))
    sway_y = int(p["strength"]*0.3*np.cos(t*p["speed"]*0.5))
    s = p["strength"]*0.6 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out = _roll_rows(frame, s)
    return np.roll(np.roll(out, sway_x, axis=1), sway_y, axis=0)

def motion_ripple(frame, t, p):
    """55. Motion Ripple — motion-blur streak + ripple."""
    h, w   = frame.shape[:2]
    rows   = np.arange(h, dtype=np.float32)
    s = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out    = _roll_rows(frame, s)
    kernel = np.zeros((1, max(3, p["wave_len"]//6)), dtype=np.float32)
    kernel[0, :] = 1.0 / kernel.shape[1]
    return _norm(cv2.filter2D(out, -1, kernel))

def drift_ripple(frame, t, p):
    """56. Drift Ripple — slow lateral drift + vertical ripple."""
    h, w  = frame.shape[:2]
    rows  = np.arange(h, dtype=np.float32)
    drift = int(t * p["speed"] * w * 0.05) % w
    s = p["strength"]*0.7 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out = _roll_rows(frame, s)
    return np.roll(out, drift, axis=1)

def float_ripple(frame, t, p):
    """57. Float Ripple — gentle vertical float + surface ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    float_y = int(p["strength"]*0.5*np.sin(t*p["speed"]*0.5))
    s = p["strength"]*0.5 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out = _roll_rows(frame, s)
    return np.roll(out, float_y, axis=0)

def hover_ripple(frame, t, p):
    """58. Hover Ripple — subtle up-down bob + fine surface texture."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    bob  = int(p["strength"]*0.3*np.sin(t*p["speed"]*1.5))
    s    = p["strength"]*0.35 * np.sin(2*np.pi*(rows/(p["wave_len"]*0.5) + t*p["speed"]))
    out  = _roll_rows(frame, s)
    return np.roll(out, bob, axis=0)

def swing_ripple(frame, t, p):
    """59. Swing Ripple — pendulum arc displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    arc  = np.sin(t*p["speed"]) * (rows/h)
    s    = p["strength"] * arc + p["strength"]*0.3*np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    return _roll_rows(frame, s)

def bounce_ripple(frame, t, p):
    """60. Bounce Ripple — gravity-drop bounce with surface ripple."""
    h, w   = frame.shape[:2]
    bounce = abs(np.sin(t*p["speed"]*np.pi))
    drop_y = int((1 - bounce) * h * 0.08)
    rows   = np.arange(h, dtype=np.float32)
    s = p["strength"]*bounce * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out = _roll_rows(frame, s)
    return np.roll(out, drop_y, axis=0)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 7 — VISUAL RIPPLE (61-70) ══════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def rgb_ripple(frame, t, p):
    """61. RGB Ripple — independent phase-shifted ripple per channel."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    out  = frame.copy()
    for c, phase in enumerate([0, np.pi*2/3, np.pi*4/3]):
        s = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]) + phase)
        ch = frame[..., c].copy()
        for y, sh in enumerate(s):
            out[y, :, c] = np.roll(ch[y], int(sh))
    return out

def neon_ripple(frame, t, p):
    """62. Neon Ripple — ripple + oversaturated edge glow."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    edges = cv2.Laplacian(out.astype(np.uint8), cv2.CV_32F)
    edges = np.abs(edges)
    neon  = np.zeros_like(out)
    neon[...,0] = edges[...,0] * 2
    neon[...,2] = edges[...,2] * 2
    return _norm(out + neon)

def led_ripple(frame, t, p):
    """63. LED Ripple — pixelated grid + per-cell brightness ripple."""
    h, w  = frame.shape[:2]
    cell  = max(4, p["wave_len"]//8)
    out   = frame.copy()
    for y in range(0, h, cell):
        for x in range(0, w, cell):
            bright = 0.7 + 0.3*np.sin(2*np.pi*(y/p["wave_len"] + x/p["wave_len"] + t*p["speed"]))
            out[y:y+cell, x:x+cell] = frame[y:y+cell, x:x+cell] * bright
    return _norm(out)

def hologram_ripple(frame, t, p):
    """64. Hologram Ripple — scan-line shift + cyan tint + ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    scan = np.ones((h,w,3), dtype=np.float32)
    scan[::3] = 0.5
    out = out * scan
    out[...,1] = np.clip(out[...,1] * 1.3, 0, 255)
    out[...,2] = np.clip(out[...,2] * 1.5, 0, 255)
    return _norm(out)

def glass_ripple(frame, t, p):
    """65. Glass Ripple — refraction + mild blur (frosted glass)."""
    h, w  = frame.shape[:2]
    X, Y  = _meshgrid(h, w)
    dx = p["strength"] * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]))
    dy = p["strength"] * np.cos(2*np.pi*(X/p["wave_len"] + t*p["speed"]*0.7))
    warped = _remap(frame, X+dx, Y+dy)
    return _norm(cv2.GaussianBlur(warped, (5,5), 1))

def crystal_ripple(frame, t, p):
    """66. Crystal Ripple — faceted angular dispersion + ripple."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    angle = np.arctan2(Y - h//2, X - w//2)
    facet = np.floor(angle / (np.pi/6)) * (np.pi/6)
    dx    = p["strength"] * np.sin(facet + t*p["speed"])
    dy    = p["strength"] * np.cos(facet + t*p["speed"])
    return _remap(frame, X+dx, Y+dy)

def reflection_ripple(frame, t, p):
    """67. Reflection Ripple — mirrored bottom half with water ripple."""
    h, w = frame.shape[:2]
    out  = frame.copy()
    half = h // 2
    mirror = frame[:half][::-1].copy()
    rows   = np.arange(half, dtype=np.float32)
    s = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    mirror = _roll_rows(mirror, s)
    out[half:half+mirror.shape[0]] = mirror[:h-half]
    return out

def refraction_ripple(frame, t, p):
    """68. Refraction Ripple — Snell's-law-inspired angle bend."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    n    = 1 + 0.3 * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]))
    dx   = p["strength"] * (n - 1) * np.cos(Y/p["wave_len"])
    dy   = p["strength"] * (n - 1) * np.sin(X/p["wave_len"])
    return _remap(frame, X+dx, Y+dy)

def prism_ripple(frame, t, p):
    """69. Prism Ripple — chromatic split + angle-based separation."""
    h, w  = frame.shape[:2]
    X, Y  = _meshgrid(h, w)
    out   = frame.copy()
    for c, shift_mul in enumerate([1.0, 0.0, -1.0]):
        dx  = p["strength"]*shift_mul * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]))
        ch  = frame[..., c].astype(np.float32)
        out[...,c] = cv2.remap(ch, (X+dx).astype(np.float32), Y,
                                cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return _norm(out)

def glow_ripple(frame, t, p):
    """70. Glow Ripple — bloom overlay on displaced frame."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    glow = cv2.GaussianBlur(out, (21,21), 8)
    return _norm(out * 0.75 + glow * 0.4)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 8 — FIRE & AIR RIPPLE (71-80) ══════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def fire_ripple(frame, t, p):
    """71. Fire Ripple — upward turbulent heat columns."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    flicker = 1 + 0.3*np.random.rand()
    s  = p["strength"]*flicker * np.sin(2*np.pi*(rows/(p["wave_len"]*0.7) - t*p["speed"]*1.5))
    s += p["strength"]*0.4 * np.sin(2*np.pi*(rows/(p["wave_len"]*0.3) - t*p["speed"]*2.5))
    out = _roll_rows(frame, s)
    out[...,0] = np.clip(out[...,0]*1.2, 0, 255)
    out[...,1] = np.clip(out[...,1]*0.8, 0, 255)
    out[...,2] = np.clip(out[...,2]*0.5, 0, 255)
    return out

def heat_ripple(frame, t, p):
    """72. Heat Ripple — thermal shimmer (rising hot air)."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    heat_row = h - rows  # strongest at bottom
    s = p["strength"] * (heat_row/h) * np.sin(2*np.pi*(rows/(p["wave_len"]*0.6) - t*p["speed"]*1.2))
    s += p["strength"]*0.2 * (heat_row/h) * np.random.randn(h)
    return _roll_rows(frame, s)

def smoke_ripple(frame, t, p):
    """73. Smoke Ripple — slow drifting tendrils."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = p["strength"]*0.5 * np.sin(2*np.pi*(Y/(p["wave_len"]*2) + t*p["speed"]*0.3)
                                    + np.cos(X/(p["wave_len"]*3)))
    dy = p["strength"]*0.3 * np.cos(2*np.pi*(X/(p["wave_len"]*2) - t*p["speed"]*0.2))
    out = _remap(frame, X+dx, Y+dy)
    gray = out.mean(axis=2, keepdims=True)
    return _norm(out*0.6 + gray*0.4)

def air_ripple(frame, t, p):
    """74. Air Ripple — near-invisible fast shimmer."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    dx = p["strength"]*0.3 * np.sin(2*np.pi*(Y/(p["wave_len"]*0.8) + t*p["speed"]*1.5))
    dy = p["strength"]*0.2 * np.cos(2*np.pi*(X/(p["wave_len"]*0.8) + t*p["speed"]*1.2))
    return _remap(frame, X+dx, Y+dy)

def wind_ripple(frame, t, p):
    """75. Wind Ripple — horizontal gust displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    gust  = np.sin(t*p["speed"]*0.4)*0.5 + 0.5
    drift = int(gust * p["strength"] * 0.5)
    s     = p["strength"]*0.5 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out   = _roll_rows(frame, s)
    return np.roll(out, drift, axis=1)

def flow_ripple(frame, t, p):
    """76. Flow Ripple — curl-field flow simulation."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    phi  = np.sin(2*np.pi*(X/p["wave_len"] + t*p["speed"])) \
         + np.cos(2*np.pi*(Y/p["wave_len"] + t*p["speed"]*0.8))
    dx   = p["strength"] * np.gradient(phi, axis=1)
    dy   = -p["strength"] * np.gradient(phi, axis=0)
    return _remap(frame, X+dx, Y+dy)

def steam_ripple(frame, t, p):
    """77. Steam Ripple — rising wisps with brightness wash."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    rise = (h - rows) / h
    s    = p["strength"]*0.6*rise * np.sin(2*np.pi*(rows/(p["wave_len"]*0.8) - t*p["speed"]))
    out  = _roll_rows(frame, s)
    fog  = np.ones_like(out) * 255
    alpha = 0.08 * rise[:, None, None]
    return _norm(out*(1-alpha) + fog*alpha)

def flame_ripple(frame, t, p):
    """78. Flame Ripple — chaotic fire tongue displacement."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    rng  = np.random.default_rng(int(t*20))
    noise = rng.uniform(-1, 1, h)
    noise = cv2.GaussianBlur(noise.reshape(-1,1).astype(np.float32),(1,11),4).flatten()
    s = p["strength"] * ((h-rows)/h) * (np.sin(2*np.pi*(rows/(p["wave_len"]*0.5) - t*p["speed"]*1.8)) + 0.3*noise)
    out = _roll_rows(frame, s)
    out[...,0] = np.clip(out[...,0]*1.3, 0, 255)
    return out

def ember_ripple(frame, t, p):
    """79. Ember Ripple — sparse bright particle upward drift."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"]*0.7 * np.sin(2*np.pi*(rows/(p["wave_len"]*0.7) - t*p["speed"]*1.3))
    out  = _roll_rows(frame, s)
    rng  = np.random.default_rng(int(t*5))
    sparks = rng.random((h, w)) > 0.998
    out[sparks] = np.clip(out[sparks] + 120, 0, 255)
    return out

def thermal_ripple(frame, t, p):
    """80. Thermal Ripple — temperature-map gradient warp."""
    h, w  = frame.shape[:2]
    X, Y  = _meshgrid(h, w)
    temp  = (frame[...,0].astype(np.float32)/255 - 0.5)
    dx    = p["strength"] * temp * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]))
    dy    = p["strength"] * temp * np.cos(2*np.pi*(X/p["wave_len"] + t*p["speed"]))
    return _remap(frame, X+dx, Y+dy)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 9 — SCI-FI RIPPLE (81-90) ══════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def gravity_ripple(frame, t, p):
    """81. Gravity Ripple — gravitational lensing around centre."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    lens = p["strength"] * 200 / (D + 50)
    lens *= np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"]))
    mx = X + lens*np.cos(ang)
    my = Y + lens*np.sin(ang)
    return _remap(frame, mx, my)

def space_ripple(frame, t, p):
    """82. Space Ripple — spacetime fabric warp."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    fabric = p["strength"] * np.sin(2*np.pi*(D/(p["wave_len"]*2) - t*p["speed"]*0.3))
    curl   = p["strength"]*0.3 * np.cos(2*np.pi*(ang/(np.pi) - t*p["speed"]*0.1))
    mx = X + (fabric+curl)*np.cos(ang)
    my = Y + (fabric+curl)*np.sin(ang)
    out = _remap(frame, mx, my)
    out[...,2] = np.clip(out[...,2]*1.2, 0, 255)
    return out

def time_ripple(frame, t, p):
    """83. Time Ripple — temporal echo smear (past frames blended)."""
    h, w  = frame.shape[:2]
    rows  = np.arange(h, dtype=np.float32)
    out   = frame.copy().astype(np.float32)
    for lag in [0.1, 0.2, 0.3]:
        s   = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + (t-lag)*p["speed"]))
        past = _roll_rows(frame, s).astype(np.float32)
        out  = out * 0.75 + past * 0.25
    return _norm(out)

def warp_ripple(frame, t, p):
    """84. Warp Ripple — warp-speed radial stretch."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    warp = 1 + 0.3*np.sin(2*np.pi*(D/p["wave_len"] - t*p["speed"]))
    mx = w/2 + (D*warp)*np.cos(ang)
    my = h/2 + (D*warp)*np.sin(ang)
    return _remap(frame, mx, my)

def portal_ripple(frame, t, p):
    """85. Portal Ripple — rotating spiral gateway."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    spiral = ang + D/p["wave_len"]*0.5 - t*p["speed"]
    mx = w/2 + D*np.cos(spiral)
    my = h/2 + D*np.sin(spiral)
    out = _remap(frame, mx, my)
    edge = np.clip(1 - D/(max(h,w)*0.5), 0, 1)
    out[...,2] = np.clip(out[...,2] + edge*40, 0, 255)
    return out

def dimension_ripple(frame, t, p):
    """86. Dimension Ripple — 3D perspective fold simulation."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    persp = 1 + 0.4*(Y/h) * np.sin(2*np.pi*t*p["speed"]*0.5)
    dx    = p["strength"] * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"])) * persp
    return _remap(frame, X+dx, Y)

def cosmic_ripple(frame, t, p):
    """87. Cosmic Ripple — nebula-like multi-scale interference."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    v1 = np.sin(2*np.pi*(X/(p["wave_len"]*2)     + t*p["speed"]*0.3))
    v2 = np.sin(2*np.pi*(Y/(p["wave_len"]*1.5)   + t*p["speed"]*0.5 + np.pi/4))
    v3 = np.sin(2*np.pi*((X-Y)/(p["wave_len"]*3) + t*p["speed"]*0.2))
    dx = p["strength"] * (v1 + v2*0.5 + v3*0.25)
    dy = p["strength"] * (v2 + v1*0.5 + v3*0.25)
    out = _remap(frame, X+dx, Y+dy)
    out[...,2] = np.clip(out[...,2]*1.3, 0, 255)
    out[...,0] = np.clip(out[...,0]*0.8, 0, 255)
    return out

def galaxy_ripple(frame, t, p):
    """88. Galaxy Ripple — spiral arm rotation."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    arm_angle = ang + D/(p["wave_len"]*2)
    spin      = t * p["speed"] * 0.5
    mx = w/2 + D*np.cos(arm_angle + spin)
    my = h/2 + D*np.sin(arm_angle + spin)
    return _remap(frame, mx, my)

def blackhole_ripple(frame, t, p):
    """89. Blackhole Ripple — extreme inward lensing singularity."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    D   = np.maximum(D, 1)
    pull = p["strength"] * 500 / (D**1.5 + 100)
    spin = t * p["speed"] * 0.8
    mx   = w/2 + D*np.cos(ang - spin) - pull*np.cos(ang)
    my   = h/2 + D*np.sin(ang - spin) - pull*np.sin(ang)
    return _remap(frame, mx, my)

def wormhole_ripple(frame, t, p):
    """90. Wormhole Ripple — inward tunnel warp (edge-to-centre zoom)."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    D, ang = _radial(h, w)
    max_r  = max(h, w) * 0.5
    r_norm = D / max_r
    tunnel = 1 + p["strength"] * 0.005 * np.sin(2*np.pi*(r_norm*3 - t*p["speed"]))
    mx = w/2 + D*tunnel*np.cos(ang)
    my = h/2 + D*tunnel*np.sin(ang)
    return _remap(frame, mx, my)

# ─────────────────────────────────────────────────────────────────────────────
# ══ GROUP 10 — STYLIZED RIPPLE (91-100) ══════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

def trippy_ripple(frame, t, p):
    """91. Trippy Ripple — kaleidoscopic colour-phase ripple."""
    h, w = frame.shape[:2]
    X, Y = _meshgrid(h, w)
    out  = frame.copy()
    for c, off in enumerate([0, 2*np.pi/3, 4*np.pi/3]):
        dx = p["strength"] * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"] + off))
        ch = frame[..., c].astype(np.float32)
        out[..., c] = cv2.remap(ch, (X+dx).astype(np.float32), Y,
                                 cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return _norm(out)

def psychedelic_ripple(frame, t, p):
    """92. Psychedelic Ripple — hue-rotated kaleidoscope warp."""
    h, w  = frame.shape[:2]
    X, Y  = _meshgrid(h, w)
    dx = p["strength"] * np.sin(2*np.pi*(Y/p["wave_len"] + t*p["speed"]))
    dy = p["strength"] * np.cos(2*np.pi*(X/p["wave_len"] + t*p["speed"]*0.7))
    out  = _remap(frame, X+dx, Y+dy)
    hsv  = cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[...,0] = (hsv[...,0] + t*30*p["speed"]) % 180
    return _norm(cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32))

def dream_ripple(frame, t, p):
    """93. Dream Ripple — soft blur + pastel tint + gentle sway."""
    h, w  = frame.shape[:2]
    rows  = np.arange(h, dtype=np.float32)
    s     = p["strength"]*0.5 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]*0.4))
    out   = _roll_rows(frame, s)
    out   = cv2.GaussianBlur(out, (9,9), 3)
    pastel = np.array([220, 200, 255], dtype=np.float32)
    return _norm(out*0.8 + pastel*0.2)

def cartoon_ripple(frame, t, p):
    """94. Cartoon Ripple — posterised + bold-outline ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    # posterise
    levels = 5
    out = (np.floor(out / (255/levels)) * (255/levels))
    edges = cv2.Canny(out.astype(np.uint8), 40, 80)
    out[edges > 0] = 20
    return _norm(out)

def anime_ripple(frame, t, p):
    """95. Anime Ripple — speed-line style + sharp ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s = p["strength"]*1.2 * np.sin(2*np.pi*(rows/(p["wave_len"]*0.5) + t*p["speed"]*1.5))
    out = _roll_rows(frame, s)
    out[::4] = np.clip(out[::4]*1.4, 0, 255)
    return out

def retro_ripple(frame, t, p):
    """96. Retro Ripple — 8-bit colour depth + scan lines + ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    # 4-bit colour
    out  = (np.floor(out / 16) * 16)
    out[::2] = out[::2] * 0.7
    return _norm(out)

def vhs_ripple(frame, t, p):
    """97. VHS Ripple — tape-tracking error + colour bleed."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    if random.random() < 0.15:
        y0 = random.randint(0, h-20)
        out[y0:y0+10] = np.roll(out[y0:y0+10], random.randint(-15,15), axis=1)
    # colour bleed
    out[...,0] = np.roll(out[...,0], 3, axis=1)
    out[...,2] = np.roll(out[...,2], -3, axis=1)
    return out

def crt_ripple(frame, t, p):
    """98. CRT Ripple — phosphor glow + horizontal scan bands."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    scan = np.ones((h,1,1), dtype=np.float32)
    scan[::2] = 0.78
    out = out * scan
    glow = cv2.GaussianBlur(out, (3,3), 1)
    return _norm(out*0.85 + glow*0.2)

def pixel_ripple(frame, t, p):
    """99. Pixel Ripple — pixelated block displacement."""
    h, w    = frame.shape[:2]
    block   = max(4, p["wave_len"]//6)
    small   = cv2.resize(frame, (w//block, h//block), interpolation=cv2.INTER_NEAREST)
    pixelated = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    rows    = np.arange(h, dtype=np.float32)
    s = p["strength"] * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    return _roll_rows(pixelated, s)

def glitch_ripple(frame, t, p):
    """100. Glitch Ripple — digital corruption + RGB tear + ripple."""
    h, w = frame.shape[:2]
    rows = np.arange(h, dtype=np.float32)
    s    = p["strength"]*0.7 * np.sin(2*np.pi*(rows/p["wave_len"] + t*p["speed"]))
    out  = _roll_rows(frame, s)
    # glitch slices
    rng  = np.random.default_rng(int(t*30))
    for _ in range(3):
        y0 = int(rng.uniform(0, h-20))
        sh = int(rng.uniform(-p["strength"]*1.5, p["strength"]*1.5))
        sl = int(rng.integers(5,25))
        out[y0:y0+sl] = np.roll(out[y0:y0+sl], sh, axis=1)
    # RGB channel tear
    out[...,0] = np.roll(out[...,0], int(rng.uniform(-4,4)), axis=1)
    out[...,2] = np.roll(out[...,2], int(rng.uniform(-4,4)), axis=1)
    return out

# ─────────────────────────────────────────────────────────────────────────────
# MASTER DISPATCH TABLE
# ─────────────────────────────────────────────────────────────────────────────

EFFECTS = {
    # id  : (function,  display_name,            group)
     1: (ripple,              "Ripple",              "Basic"),
     2: (gentle_ripple,       "Gentle Ripple",       "Basic"),
     3: (strong_ripple,       "Strong Ripple",       "Basic"),
     4: (soft_ripple,         "Soft Ripple",         "Basic"),
     5: (hard_ripple,         "Hard Ripple",         "Basic"),
     6: (smooth_ripple,       "Smooth Ripple",       "Basic"),
     7: (sharp_ripple,        "Sharp Ripple",        "Basic"),
     8: (random_ripple,       "Random Ripple",       "Basic"),
     9: (multi_ripple,        "Multi Ripple",        "Basic"),
    10: (continuous_ripple,   "Continuous Ripple",   "Basic"),
    11: (water_ripple,        "Water Ripple",        "Water"),
    12: (pond_ripple,         "Pond Ripple",         "Water"),
    13: (ocean_ripple,        "Ocean Ripple",        "Water"),
    14: (lake_ripple,         "Lake Ripple",         "Water"),
    15: (river_ripple,        "River Ripple",        "Water"),
    16: (rain_ripple,         "Rain Ripple",         "Water"),
    17: (droplet_ripple,      "Droplet Ripple",      "Water"),
    18: (splash_ripple,       "Splash Ripple",       "Water"),
    19: (bubble_ripple,       "Bubble Ripple",       "Water"),
    20: (wet_ripple,          "Wet Ripple",          "Water"),
    21: (ripple_warp,         "Ripple Warp",         "Distortion"),
    22: (ripple_distortion,   "Ripple Distortion",   "Distortion"),
    23: (wave_distortion,     "Wave Distortion",     "Distortion"),
    24: (turbulent_ripple,    "Turbulent Ripple",    "Distortion"),
    25: (twisted_ripple,      "Twisted Ripple",      "Distortion"),
    26: (curved_ripple,       "Curved Ripple",       "Distortion"),
    27: (swirl_ripple,        "Swirl Ripple",        "Distortion"),
    28: (liquid_ripple,       "Liquid Ripple",       "Distortion"),
    29: (jelly_ripple,        "Jelly Ripple",        "Distortion"),
    30: (elastic_ripple,      "Elastic Ripple",      "Distortion"),
    31: (energy_ripple,       "Energy Ripple",       "Energy"),
    32: (plasma_ripple,       "Plasma Ripple",       "Energy"),
    33: (electric_ripple,     "Electric Ripple",     "Energy"),
    34: (lightning_ripple,    "Lightning Ripple",    "Energy"),
    35: (force_ripple,        "Force Ripple",        "Energy"),
    36: (power_ripple,        "Power Ripple",        "Energy"),
    37: (shockwave_ripple,    "Shockwave Ripple",    "Energy"),
    38: (pulse_ripple,        "Pulse Ripple",        "Energy"),
    39: (quantum_ripple,      "Quantum Ripple",      "Energy"),
    40: (aura_ripple,         "Aura Ripple",         "Energy"),
    41: (sonic_ripple,        "Sonic Ripple",        "Sonic"),
    42: (audio_ripple,        "Audio Ripple",        "Sonic"),
    43: (beat_ripple,         "Beat Ripple",         "Sonic"),
    44: (music_ripple,        "Music Ripple",        "Sonic"),
    45: (frequency_ripple,    "Frequency Ripple",    "Sonic"),
    46: (bass_ripple,         "Bass Ripple",         "Sonic"),
    47: (echo_ripple,         "Echo Ripple",         "Sonic"),
    48: (vibration_ripple,    "Vibration Ripple",    "Sonic"),
    49: (resonance_ripple,    "Resonance Ripple",    "Sonic"),
    50: (soundwave_ripple,    "Soundwave Ripple",    "Sonic"),
    51: (shaky_ripple,        "Shaky Ripple",        "Motion"),
    52: (wiggle_ripple,       "Wiggle Ripple",       "Motion"),
    53: (jitter_ripple,       "Jitter Ripple",       "Motion"),
    54: (camera_ripple,       "Camera Ripple",       "Motion"),
    55: (motion_ripple,       "Motion Ripple",       "Motion"),
    56: (drift_ripple,        "Drift Ripple",        "Motion"),
    57: (float_ripple,        "Float Ripple",        "Motion"),
    58: (hover_ripple,        "Hover Ripple",        "Motion"),
    59: (swing_ripple,        "Swing Ripple",        "Motion"),
    60: (bounce_ripple,       "Bounce Ripple",       "Motion"),
    61: (rgb_ripple,          "RGB Ripple",          "Visual"),
    62: (neon_ripple,         "Neon Ripple",         "Visual"),
    63: (led_ripple,          "LED Ripple",          "Visual"),
    64: (hologram_ripple,     "Hologram Ripple",     "Visual"),
    65: (glass_ripple,        "Glass Ripple",        "Visual"),
    66: (crystal_ripple,      "Crystal Ripple",      "Visual"),
    67: (reflection_ripple,   "Reflection Ripple",   "Visual"),
    68: (refraction_ripple,   "Refraction Ripple",   "Visual"),
    69: (prism_ripple,        "Prism Ripple",        "Visual"),
    70: (glow_ripple,         "Glow Ripple",         "Visual"),
    71: (fire_ripple,         "Fire Ripple",         "Fire & Air"),
    72: (heat_ripple,         "Heat Ripple",         "Fire & Air"),
    73: (smoke_ripple,        "Smoke Ripple",        "Fire & Air"),
    74: (air_ripple,          "Air Ripple",          "Fire & Air"),
    75: (wind_ripple,         "Wind Ripple",         "Fire & Air"),
    76: (flow_ripple,         "Flow Ripple",         "Fire & Air"),
    77: (steam_ripple,        "Steam Ripple",        "Fire & Air"),
    78: (flame_ripple,        "Flame Ripple",        "Fire & Air"),
    79: (ember_ripple,        "Ember Ripple",        "Fire & Air"),
    80: (thermal_ripple,      "Thermal Ripple",      "Fire & Air"),
    81: (gravity_ripple,      "Gravity Ripple",      "Sci-Fi"),
    82: (space_ripple,        "Space Ripple",        "Sci-Fi"),
    83: (time_ripple,         "Time Ripple",         "Sci-Fi"),
    84: (warp_ripple,         "Warp Ripple",         "Sci-Fi"),
    85: (portal_ripple,       "Portal Ripple",       "Sci-Fi"),
    86: (dimension_ripple,    "Dimension Ripple",    "Sci-Fi"),
    87: (cosmic_ripple,       "Cosmic Ripple",       "Sci-Fi"),
    88: (galaxy_ripple,       "Galaxy Ripple",       "Sci-Fi"),
    89: (blackhole_ripple,    "Blackhole Ripple",    "Sci-Fi"),
    90: (wormhole_ripple,     "Wormhole Ripple",     "Sci-Fi"),
    91: (trippy_ripple,       "Trippy Ripple",       "Stylized"),
    92: (psychedelic_ripple,  "Psychedelic Ripple",  "Stylized"),
    93: (dream_ripple,        "Dream Ripple",        "Stylized"),
    94: (cartoon_ripple,      "Cartoon Ripple",      "Stylized"),
    95: (anime_ripple,        "Anime Ripple",        "Stylized"),
    96: (retro_ripple,        "Retro Ripple",        "Stylized"),
    97: (vhs_ripple,          "VHS Ripple",          "Stylized"),
    98: (crt_ripple,          "CRT Ripple",          "Stylized"),
    99: (pixel_ripple,        "Pixel Ripple",        "Stylized"),
   100: (glitch_ripple,       "Glitch Ripple",       "Stylized"),
}

GROUP_RANGES = {
    "Basic":      range(1,  11),
    "Water":      range(11, 21),
    "Distortion": range(21, 31),
    "Energy":     range(31, 41),
    "Sonic":      range(41, 51),
    "Motion":     range(51, 61),
    "Visual":     range(61, 71),
    "Fire & Air": range(71, 81),
    "Sci-Fi":     range(81, 91),
    "Stylized":   range(91, 101),
}

def apply_ripple_effect(frame, effect_id, t, strength=8, wave_len=120, speed=2.0):
    """Main public API: apply any of the 100 effects by ID."""
    if effect_id not in EFFECTS:
        raise ValueError(f"Effect ID must be 1-100, got {effect_id}")
    fn, name, group = EFFECTS[effect_id]
    p = {"strength": strength, "wave_len": max(10, wave_len), "speed": speed}
    result = fn(frame.astype(np.float32), t, p)
    return np.clip(result, 0, 255).astype(np.float32)

'''

exec(compile(EFFECTS_SRC, '<ripple_engine>', 'exec'), globals())
print(f'✅ Loaded {len(EFFECTS)} ripple effects across {len(GROUP_RANGES)} groups')
for gname, grange in GROUP_RANGES.items():
    names = [EFFECTS[i][1] for i in grange]
    print(f'  {gname}: {names}')


### Cell 3 · Launch the control panel
① Pick a group  →  ② Pick an effect  →  ③ Adjust parameters  →  ④ Click **▶ Upload & Render**

In [ ]:

import sys, types, cv2, numpy as np, os, subprocess, shutil
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files as colab_files

# Effects already loaded into globals() by Cell 2
import math  # FIX: explicit import in case Cell 2 exec scope differs

# ── utils ────────────────────────────────────────────────────
def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: raise ValueError(f"Cannot load: {path}")
    if img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA)
    elif img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32)

def resize_keep_aspect(img, tw, th):
    h, w = img.shape[:2]
    sc = min(tw/w, th/h)
    return cv2.resize(img, (int(w*sc), int(h*sc)), interpolation=cv2.INTER_AREA)

def center_on_canvas(img, cw, ch, bg=(0,0,0)):
    canvas = np.full((ch, cw, 3), bg, dtype=np.float32)
    h, w = img.shape[:2]
    x0, y0 = (cw-w)//2, (ch-h)//2
    if img.shape[2] == 4:
        rgb, a = img[...,:3], img[...,3:4]/255.0
        canvas[y0:y0+h, x0:x0+w] = rgb*a + canvas[y0:y0+h, x0:x0+w]*(1-a)
    else:
        canvas[y0:y0+h, x0:x0+w] = img
    return canvas

def normalize_frame(f): return np.clip(f, 0, 255).astype(np.uint8)

WATERMARK_DATA = {"img": None, "text": None}

def apply_watermark(frame_u8, wm_cfg):
    out = frame_u8.copy().astype(np.float32)
    h, w = out.shape[:2]
    if WATERMARK_DATA["img"] is not None:
        wm = WATERMARK_DATA["img"].copy()
        sc = wm_cfg["scale"]
        nw, nh = max(1,int(wm.shape[1]*sc)), max(1,int(wm.shape[0]*sc))
        wm = cv2.resize(wm, (nw,nh), interpolation=cv2.INTER_AREA)
        pad = wm_cfg["padding"]
        pos = wm_cfg["position"]
        lut = {"Top-Left":(pad,pad),"Top-Right":(w-nw-pad,pad),
               "Bottom-Left":(pad,h-nh-pad),"Bottom-Right":(w-nw-pad,h-nh-pad),
               "Center":((w-nw)//2,(h-nh)//2)}
        x0,y0 = lut.get(pos,(pad,pad))
        x0,y0 = max(0,min(x0,w-nw)), max(0,min(y0,h-nh))
        alpha = wm_cfg["opacity"]
        roi = out[y0:y0+nh, x0:x0+nw]
        if wm.shape[2] == 4:
            wa = wm[...,3:4]/255.0*alpha
            out[y0:y0+nh, x0:x0+nw] = roi*(1-wa) + wm[...,:3]*wa
        else:
            out[y0:y0+nh, x0:x0+nw] = roi*(1-alpha) + wm[...,:3]*alpha
    elif WATERMARK_DATA["text"]:
        cfg = WATERMARK_DATA["text"]
        (tw,th),_ = cv2.getTextSize(cfg["text"], cv2.FONT_HERSHEY_SIMPLEX, cfg["scale"], cfg["thick"])
        pad=30
        lut = {"Top-Left":(pad,pad+th),"Top-Right":(w-tw-pad,pad+th),
               "Bottom-Left":(pad,h-pad),"Bottom-Right":(w-tw-pad,h-pad),
               "Center":((w-tw)//2,(h+th)//2)}
        x0,y0 = lut.get(cfg["pos"],(pad,h-pad))
        ol = out.copy()
        c  = (cfg["color"][2], cfg["color"][1], cfg["color"][0])
        cv2.putText(ol, cfg["text"], (x0,y0), cv2.FONT_HERSHEY_SIMPLEX,
                    cfg["scale"], c, cfg["thick"], cv2.LINE_AA)
        out = cv2.addWeighted(ol, cfg["opacity"], out, 1-cfg["opacity"], 0)
    return np.clip(out,0,255).astype(np.uint8)

def render_video(image_path, vid, ripple, wm_cfg, audio_cfg):
    img        = load_image(image_path)
    img        = resize_keep_aspect(img, vid["w"], vid["h"])
    base_frame = center_on_canvas(img, vid["w"], vid["h"], vid["bg"])
    fourcc     = cv2.VideoWriter_fourcc(*"mp4v")
    out_path   = vid["output"]
    vw         = cv2.VideoWriter(out_path, fourcc, vid["fps"], (vid["w"], vid["h"]))
    total      = int(vid["fps"] * vid["duration"])
    t_on, t_off = ripple["start"], ripple["end"]
    print(f"⚙️  Rendering {total} frames  ·  Effect #{ripple['id']}: {ripple['name']}")
    for i in tqdm(range(total)):
        t     = i / vid["fps"]
        frame = base_frame.copy()
        if t_on <= t <= t_off:
            frame = apply_ripple_effect(frame, ripple["id"], t,
                        ripple["strength"], ripple["wave_len"], ripple["speed"])
        frame = normalize_frame(frame)
        frame = apply_watermark(frame, wm_cfg)
        vw.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    vw.release()
    print(f"✅  Video saved → {out_path}")
    # audio
    if audio_cfg["enabled"] and audio_cfg["path"]:
        try:
            if shutil.which("ffmpeg") is None:
                subprocess.run(["apt-get","install","-y","-q","ffmpeg"], check=True, capture_output=True)
            with_a = "output_with_audio.mp4"
            vol, a_st, fade, dur = audio_cfg["volume"], audio_cfg["start"], audio_cfg["fade"], vid["duration"]
            af = f"adelay={int(a_st*1000)}|{int(a_st*1000)},volume={vol}"
            if fade: af += f",afade=t=out:st={max(0,dur-2)}:d=2"
            res = subprocess.run(["ffmpeg","-y","-i",out_path,"-i",audio_cfg["path"],
                "-filter_complex",f"[1:a]{af}[aout]","-map","0:v","-map","[aout]",
                "-c:v","copy","-c:a","aac","-b:a","192k","-shortest",with_a],
                capture_output=True, text=True)
            if res.returncode == 0:
                os.replace(with_a, out_path); print(f"🎵  Audio merged")
            else: print("❌  ffmpeg error:", res.stderr[-300:])
        except Exception as e: print(f"⚠️  Audio failed: {e}")
    colab_files.download(out_path)

# ════════════════════════════════════════════════════════════════════════════════
# UI
# ════════════════════════════════════════════════════════════════════════════════
s   = {"description_width": "170px"}
lay = widgets.Layout(width="450px")

def hdr(txt):
    return widgets.HTML(
        f"<div style='margin:12px 0 3px;padding:5px 12px;background:#0f2744;"
        f"color:#38bdf8;font-weight:700;border-radius:6px;"
        f"font-family:monospace;font-size:12px;letter-spacing:1px'>{txt}</div>")

# ── Effect picker ─────────────────────────────────────────────
GROUP_NAMES  = ["Basic","Water","Distortion","Energy","Sonic",
                "Motion","Visual","Fire & Air","Sci-Fi","Stylized"]
GROUP_ICONS  = ["〰️","🌊","🌀","⚡","🔊","💨","🌈","🔥","🚀","🎨"]

w_group = widgets.ToggleButtons(
    options=[f"{i} {g}" for i,g in zip(GROUP_ICONS, GROUP_NAMES)],
    value=f"{GROUP_ICONS[0]} {GROUP_NAMES[0]}",
    description="Group:",
    style={"description_width":"60px","button_width":"90px"},
    layout=widgets.Layout(width="100%"))

def _group_options(group_name):
    r = GROUP_RANGES[group_name]
    return [f"#{i:02d}  {EFFECTS[i][1]}" for i in r]

w_effect_pick = widgets.Select(
    options=_group_options("Basic"), value=_group_options("Basic")[0],
    description="Effect:", rows=10, style=s,
    layout=widgets.Layout(width="450px"))

def _on_group(change):
    gname = change["new"].split(" ", 1)[1]
    opts  = _group_options(gname)
    w_effect_pick.options = opts
    w_effect_pick.value   = opts[0]

w_group.observe(_on_group, names="value")

# ── Ripple params ─────────────────────────────────────────────
w_strength = widgets.FloatSlider(value=8,  min=1,   max=60,  step=0.5,
    description="Strength:",  style=s, layout=lay)
w_wave_len = widgets.IntSlider(value=120, min=10,  max=400, step=5,
    description="Wave Length:", style=s, layout=lay)
w_speed    = widgets.FloatSlider(value=2.0, min=0.1, max=10,  step=0.1,
    description="Speed:",     style=s, layout=lay)
w_t_start  = widgets.FloatSlider(value=0.5, min=0, max=59, step=0.5,
    description="Effect Start (s):", style=s, layout=lay)
w_t_end    = widgets.FloatSlider(value=7.5, min=1, max=60, step=0.5,
    description="Effect End (s):",   style=s, layout=lay)

# ── Video settings ────────────────────────────────────────────
w_width    = widgets.Dropdown(options=[720,1080,1440], value=1080,
    description="Width:",    style=s, layout=lay)
w_height   = widgets.Dropdown(options=[1280,1920,2560], value=1920,
    description="Height:",   style=s, layout=lay)
w_fps      = widgets.Dropdown(options=[24,30,60], value=30,
    description="FPS:",      style=s, layout=lay)
w_duration = widgets.IntSlider(value=8, min=2, max=60,
    description="Duration (s):", style=s, layout=lay)
w_bg_r     = widgets.IntSlider(value=0, min=0, max=255,
    description="BG Red:",   style=s, layout=lay)
w_bg_g     = widgets.IntSlider(value=0, min=0, max=255,
    description="BG Green:", style=s, layout=lay)
w_bg_b     = widgets.IntSlider(value=0, min=0, max=255,
    description="BG Blue:",  style=s, layout=lay)
w_output   = widgets.Text(value="output.mp4",
    description="Output file:", style=s, layout=lay)

# ── Watermark ─────────────────────────────────────────────────
w_wm_type = widgets.ToggleButtons(
    options=["Image","Text","None"], value="None",
    description="Watermark:",
    style={"description_width":"90px","button_width":"70px"},
    layout=widgets.Layout(width="380px"))

w_wm_scale   = widgets.FloatSlider(value=0.18, min=0.02, max=0.8, step=0.01, description="Scale:", style=s, layout=lay)
w_wm_opacity = widgets.FloatSlider(value=0.75, min=0, max=1, step=0.05, description="Opacity:", style=s, layout=lay)
w_wm_pos     = widgets.Dropdown(options=["Top-Left","Top-Right","Bottom-Left","Bottom-Right","Center"],
    value="Bottom-Right", description="Position:", style=s, layout=lay)
w_wm_pad     = widgets.IntSlider(value=30, min=0, max=200, description="Padding:", style=s, layout=lay)

w_wm_text    = widgets.Text(value="© Your Name", description="Text:", style=s, layout=lay)
w_wm_fsize   = widgets.FloatSlider(value=1.2, min=0.3, max=4, step=0.1, description="Font Scale:", style=s, layout=lay)
w_wm_thick   = widgets.IntSlider(value=2, min=1, max=6, description="Thickness:", style=s, layout=lay)
w_wm_cr      = widgets.IntSlider(value=255, min=0, max=255, description="Text Red:", style=s, layout=lay)
w_wm_cg      = widgets.IntSlider(value=255, min=0, max=255, description="Text Green:", style=s, layout=lay)
w_wm_cb      = widgets.IntSlider(value=255, min=0, max=255, description="Text Blue:", style=s, layout=lay)
w_wm_top     = widgets.FloatSlider(value=0.7, min=0, max=1, step=0.05, description="Text Opacity:", style=s, layout=lay)
w_wm_tpos    = widgets.Dropdown(options=["Top-Left","Top-Right","Bottom-Left","Bottom-Right","Center"],
    value="Bottom-Right", description="Text Position:", style=s, layout=lay)

wm_img_box  = widgets.VBox([w_wm_scale, w_wm_opacity, w_wm_pos, w_wm_pad])
wm_txt_box  = widgets.VBox([w_wm_text, w_wm_fsize, w_wm_thick, w_wm_cr, w_wm_cg, w_wm_cb, w_wm_top, w_wm_tpos])
wm_dyn_box  = widgets.VBox([])

def _on_wm(change):
    v = change["new"]
    wm_dyn_box.children = [wm_img_box] if v=="Image" else ([wm_txt_box] if v=="Text" else [])
w_wm_type.observe(_on_wm, names="value")

# ── Audio ─────────────────────────────────────────────────────
w_audio_tog = widgets.ToggleButtons(options=["No","Yes"], value="No",
    description="Add sound?",
    style={"description_width":"90px","button_width":"60px"},
    layout=widgets.Layout(width="320px"))
w_audio_vol  = widgets.FloatSlider(value=0.8, min=0, max=1, step=0.05, description="Volume:", style=s, layout=lay)
w_audio_st   = widgets.FloatSlider(value=0, min=0, max=60, step=0.5, description="Audio delay (s):", style=s, layout=lay)
w_audio_fade = widgets.Checkbox(value=True, description="Fade out at end", style=s, layout=lay)
w_audio_box  = widgets.VBox([])

def _on_audio(change):
    w_audio_box.children = ([w_audio_vol, w_audio_st, w_audio_fade,
        widgets.HTML("<span style='color:#64748b;font-size:11px'>Audio uploaded when you click Render</span>")]
        if change["new"]=="Yes" else [])
w_audio_tog.observe(_on_audio, names="value")

# ── Render ────────────────────────────────────────────────────
w_btn    = widgets.Button(description="▶  Upload & Render",
    button_style="success", layout=widgets.Layout(width="220px", height="44px"))
w_status = widgets.Output()

def on_render(_):
    global WATERMARK_DATA
    w_status.clear_output()
    with w_status:
        # parse effect id from picker
        pick_str = w_effect_pick.value  # e.g. '#01  Ripple' or '#100  Glitch Ripple'
        eff_id   = int(pick_str[1:pick_str.index(' ')])  # FIX: parse until first space
        eff_name = EFFECTS[eff_id][1]

        vid = {"w": w_width.value, "h": w_height.value, "fps": w_fps.value,
               "duration": w_duration.value, "output": w_output.value,
               "bg": (w_bg_r.value, w_bg_g.value, w_bg_b.value)}
        ripple = {"id": eff_id, "name": eff_name,
                  "strength": w_strength.value, "wave_len": w_wave_len.value,
                  "speed": w_speed.value, "start": w_t_start.value, "end": w_t_end.value}
        wm_cfg = {"scale": w_wm_scale.value, "opacity": w_wm_opacity.value,
                  "position": w_wm_pos.value, "padding": w_wm_pad.value}

        WATERMARK_DATA["img"]  = None
        WATERMARK_DATA["text"] = None
        wm_type = w_wm_type.value
        if wm_type == "Image":
            print("🖼️  Upload watermark image (PNG recommended)…")
            up = colab_files.upload()
            if up:
                WATERMARK_DATA["img"] = load_image(list(up.keys())[0])
                print("   Watermark loaded.")
        elif wm_type == "Text":
            WATERMARK_DATA["text"] = {"text": w_wm_text.value, "scale": w_wm_fsize.value,
                "thick": w_wm_thick.value, "color": (w_wm_cr.value, w_wm_cg.value, w_wm_cb.value),
                "opacity": w_wm_top.value, "pos": w_wm_tpos.value}

        audio_cfg = {"enabled": False, "path": None,
                     "volume": w_audio_vol.value, "start": w_audio_st.value, "fade": w_audio_fade.value}
        if w_audio_tog.value == "Yes":
            print("🎵  Upload audio file (MP3/WAV/OGG/AAC)…")
            up = colab_files.upload()
            if up:
                audio_cfg["enabled"] = True
                audio_cfg["path"]    = list(up.keys())[0]

        print("📂  Upload your main image…")
        up = colab_files.upload()
        if not up:
            print("❌  No image. Cancelled.")
            return
        render_video(list(up.keys())[0], vid, ripple, wm_cfg, audio_cfg)

w_btn.on_click(on_render)

# ── Layout ────────────────────────────────────────────────────
display(HTML("""
<div style="background:linear-gradient(135deg,#0f172a,#1e293b);
     border-radius:12px;padding:20px 24px 14px;max-width:540px;margin-bottom:10px">
  <div style="font-family:monospace;font-size:20px;font-weight:800;
       color:#38bdf8;letter-spacing:3px">🌊 RIPPLE ENGINE</div>
  <div style="font-family:monospace;font-size:11px;color:#475569;margin-top:3px">
    100 effects · 10 groups · full control
  </div>
</div>"""))

display(widgets.VBox([
    hdr("① SELECT EFFECT GROUP"),
    w_group,
    hdr("② SELECT EFFECT"),
    w_effect_pick,
    hdr("③ RIPPLE PARAMETERS"),
    w_strength, w_wave_len, w_speed, w_t_start, w_t_end,
    hdr("④ VIDEO SETTINGS"),
    w_width, w_height, w_fps, w_duration, w_output,
    hdr("⑤ BACKGROUND COLOUR"),
    w_bg_r, w_bg_g, w_bg_b,
    hdr("⑥ WATERMARK"),
    w_wm_type, wm_dyn_box,
    hdr("⑦ BACKGROUND SOUND"),
    w_audio_tog, w_audio_box,
    widgets.HTML("<div style='height:10px'></div>"),
    w_btn, w_status,
]))